In [1]:
!pip -q install catboost


In [2]:
import os
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from datetime import timedelta


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Maize_Project"
print("DATA_DIR =", DATA_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/drive/MyDrive/Maize_Project


In [4]:
agri = pd.read_csv(os.path.join(DATA_DIR, "agriBORA_maize_prices.csv"))
agri_fix = pd.read_csv(os.path.join(DATA_DIR, "agriBORA_maize_prices_weeks_46_to_51.csv"))
kamis = pd.read_csv(os.path.join(DATA_DIR, "kamis_maize_prices.csv"))


agri["Date"] = pd.to_datetime(agri["Date"])
agri_fix["Date"] = pd.to_datetime(agri_fix["Date"])
kamis["Date"] = pd.to_datetime(kamis["Date"])

agri_all = pd.concat([agri, agri_fix], ignore_index=True)

TARGET_COUNTIES = ["Kiambu", "Kirinyaga", "Mombasa", "Nairobi", "Uasin-Gishu"]
agri_all.head()


,County,Date,WholeSale,Commodity_Classification,Year_Week,WeekofYear
0,Kirinyaga,2023-10-02,48.890,Dry_White_Maize,2023-40,40
1,Nairobi,2023-10-02,49.630,Dry_White_Maize,2023-40,40
2,Uasin-Gishu,2023-10-02,46.670,Dry_White_Maize,2023-40,40
3,Kisumu,2023-10-02,43.885,Dry_White_Maize,2023-40,40
4,Kiambu,2023-10-02,46.670,Dry_White_Maize,2023-40,40


In [5]:
agri_all = agri_all[
    (agri_all["Commodity_Classification"] == "Dry_White_Maize") &
    (agri_all["County"].isin(TARGET_COUNTIES))
].copy()

agri_all["weekofyear"] = agri_all["WeekofYear"].astype(int)
agri_all["month"] = agri_all["Date"].dt.month

agri_all = agri_all.sort_values(["County", "Date"]).reset_index(drop=True)


In [6]:
kamis = kamis[kamis["Commodity_Classification"] == "Dry_White_Maize"].copy()
kamis["weekofyear"] = kamis["WeekofYear"].astype(int)

kamis_w = (
    kamis.groupby(["County", "weekofyear"], as_index=False)
         .agg(
             kamis_wholesale_mean=("Wholesale", "mean"),
             kamis_retail_mean=("Retail", "mean"),
             kamis_supply_mean=("SupplyVolume", "mean")
         )
)

agri_all = agri_all.merge(
    kamis_w,
    on=["County", "weekofyear"],
    how="left"
)

for c in ["kamis_wholesale_mean", "kamis_retail_mean", "kamis_supply_mean"]:
    agri_all[c] = agri_all.groupby("County")[c].transform(lambda s: s.ffill().bfill())


In [7]:
def make_features(df):
    df = df.sort_values(["County", "Date"]).copy()

    df["log_price"] = np.log(df["WholeSale"])

    for lag in [1,2,3,4]:
        df[f"lag{lag}"] = df.groupby("County")["WholeSale"].shift(lag)

    df["roll3"] = df.groupby("County")["WholeSale"].transform(lambda s: s.rolling(3).mean())
    df["roll5"] = df.groupby("County")["WholeSale"].transform(lambda s: s.rolling(5).mean())

    df["trend"] = df["roll5"] - df["lag1"]

    df = df.dropna().reset_index(drop=True)
    return df

agri_feat = make_features(agri_all)


In [8]:
le = LabelEncoder()
agri_feat["county_id"] = le.fit_transform(agri_feat["County"])

FEATURES = [
    "county_id",
    "weekofyear",
    "month",
    "lag1","lag2","lag3","lag4",
    "roll3","roll5","trend",
    "kamis_wholesale_mean",
    "kamis_retail_mean",
    "kamis_supply_mean"
]

TARGET = "log_price"


In [9]:
X = agri_feat[FEATURES]
y = agri_feat[TARGET]

model = CatBoostRegressor(
    loss_function="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=1200,
    random_seed=42,
    verbose=200
)

model.fit(X, y)


0:	learn: 0.1241016	total: 75.6ms	remaining: 1m 30s
200:	learn: 0.0145002	total: 3.95s	remaining: 19.6s
400:	learn: 0.0043536	total: 6.55s	remaining: 13s
600:	learn: 0.0015633	total: 8.26s	remaining: 8.23s
800:	learn: 0.0006102	total: 9.31s	remaining: 4.64s
1000:	learn: 0.0002463	total: 10.3s	remaining: 2.05s
1199:	learn: 0.0000997	total: 11.3s	remaining: 0us


In [10]:
final_ids = []
for c in TARGET_COUNTIES:
    final_ids.append(f"{c}_Week_52")
for c in TARGET_COUNTIES:
    final_ids.append(f"{c}_Week_1")  # ISO week 53 saved as Week_1

submission = pd.DataFrame({"ID": final_ids})


In [11]:
# anchor week = last week common to all counties
common_weeks = set.intersection(*[
    set(agri_feat.loc[agri_feat["County"] == c, "weekofyear"])
    for c in TARGET_COUNTIES
])
anchor_week = max(common_weeks)

rows = []

for _, r in submission.iterrows():
    county = r["ID"].split("_Week_")[0]
    target_week = int(r["ID"].split("_Week_")[1])

    anchor = agri_feat[
        (agri_feat["County"] == county) &
        (agri_feat["weekofyear"] == anchor_week)
    ].iloc[-1]

    Xf = pd.DataFrame([anchor[FEATURES]])

    pred_log = model.predict(Xf)[0]
    pred = float(np.exp(pred_log))

    rows.append(pred)

submission["Target_RMSE"] = rows
submission["Target_MAE"] = rows


In [12]:
submission.to_csv("FINAL_SUBMISSION_WEEK52_WEEK1.csv", index=False)
submission


,ID,Target_RMSE,Target_MAE
0,Kiambu_Week_52,44.442967,44.442967
1,Kirinyaga_Week_52,46.668313,46.668313
2,Mombasa_Week_52,42.220958,42.220958
3,Nairobi_Week_52,42.776096,42.776096
4,Uasin-Gishu_Week_52,41.669134,41.669134
5,Kiambu_Week_1,44.442967,44.442967
6,Kirinyaga_Week_1,46.668313,46.668313
7,Mombasa_Week_1,42.220958,42.220958
8,Nairobi_Week_1,42.776096,42.776096
9,Uasin-Gishu_Week_1,41.669134,41.669134
